In [1]:
import pandas as pd
import numpy as np
from sklearn.dummy import DummyRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error

train = pd.read_csv("../data/processed/products_train_clean.csv")
test = pd.read_csv("../data/processed/products_test_clean.csv")

for d in (train, test):
    d["log_price"] = np.log1p(d["price"])
    d["text"] = d["Product Name"].fillna("") + " " + d["Description"].fillna("")
    d["sub_category"] = d["sub_category"].fillna("unknown")

features = ["text", "main_category", "sub_category"]
Xtr, ytr = train[features], train["log_price"]
Xte, yte = test[features], test["log_price"]

In [2]:
def report(name, model):
    pred = model.predict(Xte)
    r2 = r2_score(yte, pred)
    mae = mean_absolute_error(np.expm1(yte), np.expm1(pred))
    print(f"{name}: R² = {r2:.3f} | average error = ${mae:.2f}")

In [3]:
dummy = DummyRegressor(strategy="mean").fit(Xtr, ytr)
report("Baseline (always the mean)", dummy)

Baseline (always the mean): R² = -0.000 | average error = $38.13


In [4]:
pre = ColumnTransformer([
    ("text", TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=3), "text"),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["main_category", "sub_category"]),
])

model = Pipeline([("pre", pre), ("reg", Ridge(alpha=1.0))])
model.fit(Xtr, ytr)
report("Ridge + TF-IDF + categories", model)

Ridge + TF-IDF + categories: R² = 0.854 | average error = $16.74


In [5]:
names = model.named_steps["pre"].get_feature_names_out()
coefs = model.named_steps["reg"].coef_
order = np.argsort(coefs)
print("Cheaper:", names[order[:15]])
print("Pricier:", names[order[-15:]])

Cheaper: ['text__darice' 'text__candle' 'text__scale figures' 'text__yo'
 'text__unique' 'text__pcs' 'text__rattle' 'text__doll house'
 'text__laces' 'text__anti dust' 'text__25x' 'text__76'
 'text__mouse plush' 'text__the go' 'text__collect']
Pricier: ['text__kitchen' 'text__scale pvc' 'text__rink' 'text__coding' 'text__01'
 'text__statue' 'text__4wd' 'text__longer' 'text__equipment'
 'text__backyard' 'text__inflatable water' 'text__dji' 'text__enclosure'
 'text__10 times' 'text__slide']
